# AMRPA + CAM: RoBERTa on HotpotQA

Trains RoBERTa + AMRPA on preprocessed HotpotQA `.pt` files.
Uses the AMRPA library from GitHub.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
import subprocess
# Clone library from GitHub (replace with your repo URL)
# subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/amrpa.git',
#                 '/kaggle/working/amrpa-lib'])

# OR: if uploaded as Kaggle dataset
import sys
sys.path.insert(0, '/kaggle/input/amrpa-lib')  # adjust to your dataset name

import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, get_linear_schedule_with_warmup

from amrpa import (
    AMRPAConfig, AMRPAForQA,
    PreprocessedQADataset, train_epoch, evaluate, build_optimizer
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('AMRPA library loaded ✓')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
config = AMRPAConfig.for_encoder(
    d_model         = 768,
    n_heads         = 12,
    n_amrpa_layers  = 4,
    gamma           = 0.9,
    epsilon         = 0.001,
    alpha_temperature = 0.25,
    d_mlp           = 384,
    dropout         = 0.2,
    diversity_weight = 0.005,
    gate_reg_weight  = 0.05,
    label_smoothing  = 0.05,
    freeze_embeddings = True,
    freeze_layers    = -1,   # auto: freeze all but last 8
)

# Training hyperparameters
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 2e-5
DATA_DIR    = '/kaggle/input/tokenized-hotpot'
SAVE_DIR    = '/kaggle/working'

print(config)

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────
train_dataset = PreprocessedQADataset(f'{DATA_DIR}/train_processed.pt')
val_dataset   = PreprocessedQADataset(f'{DATA_DIR}/val_processed.pt')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
model = AMRPAForQA(config, model_name='roberta-base').to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
# ── Optimizer + Scheduler ─────────────────────────────────────────────────
optimizer    = build_optimizer(model, config, lr=LR)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = max(1, int(total_steps * 0.1))
scheduler    = get_linear_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────
import time, gc

history = {
    'train_loss': [], 'val_loss': [], 'val_em': [], 'val_f1': [],
    'gate_impact': [], 'alpha_diversity': [], 'memory_contribution': []
}

best_f1          = 0.0
best_val_loss    = float('inf')
patience_counter = 0
PATIENCE         = 8

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch+1}/{EPOCHS}")
    print(f"{'='*60}")
    t0 = time.time()

    train_loss, train_metrics = train_epoch(
        model, train_loader, optimizer, scheduler, device, config, epoch
    )
    val_loss, val_em, val_f1, val_metrics = evaluate(
        model, val_loader, tokenizer, device
    )

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_em'].append(val_em)
    history['val_f1'].append(val_f1)
    history['gate_impact'].append(val_metrics.get('gate_impact', 0.0))
    history['alpha_diversity'].append(val_metrics.get('alpha_diversity', 0.0))
    history['memory_contribution'].append(val_metrics.get('memory_contribution', 0.0))

    print(f"\nEpoch {epoch+1}: loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | EM={val_em:.4f} | F1={val_f1:.4f} | "
          f"time={( time.time()-t0)/60:.1f}min")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f'{SAVE_DIR}/best_amrpa_model.pt')
        print(f"  ✅ New best F1: {best_f1:.4f} → saved")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping (patience={PATIENCE})")
            break

    torch.cuda.empty_cache(); gc.collect()

print(f"\nTraining complete. Best F1: {best_f1:.4f}")

In [ ]:
# ── Plot Results ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RoBERTa + AMRPA Training Results', fontsize=14, fontweight='bold')
epochs = range(1, len(history['train_loss']) + 1)

axes[0,0].plot(epochs, history['train_loss'], 'b-o', label='Train')
axes[0,0].plot(epochs, history['val_loss'],   'r-o', label='Val')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(epochs, history['val_em'], 'g-o')
axes[0,1].set_title(f'Exact Match (Best: {max(history["val_em"]):.3f})')
axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(epochs, history['val_f1'], 'purple', marker='o')
axes[0,2].set_title(f'F1 Score (Best: {max(history["val_f1"]):.3f})')
axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(epochs, history['gate_impact'], 'orange', marker='o')
axes[1,0].fill_between(epochs, 0, history['gate_impact'], alpha=0.3, color='orange')
axes[1,0].set_title('Gate Impact (CLAIM 1)'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(epochs, history['alpha_diversity'], 'teal', marker='s')
axes[1,1].fill_between(epochs, 0, history['alpha_diversity'], alpha=0.3, color='teal')
axes[1,1].set_title('Alpha Diversity (CLAIM 2)'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(epochs, history['memory_contribution'], 'crimson', marker='^')
axes[1,2].fill_between(epochs, 0, history['memory_contribution'], alpha=0.3, color='crimson')
axes[1,2].set_title('Memory Contribution (CLAIM 4)'); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/amrpa_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')